# Experimento 010 — Bootstrap emparejado de ROC AUC

**Objetivo.** Cuantificar la incertidumbre de la diferencia de ROC AUC entre el mejor XGBoost individual y las cuatro estrategias de ensamble, sin volver a entrenar los modelos. Si se encuentran las predicciones OOF de `xgb_nested_grid`, se añade el XGBoost elegido mediante `GridSearchCV`.

**Hipótesis:** para dos modelos A y B evaluados sobre las mismas transacciones, $\Delta=\operatorname{AUC}(A)-\operatorname{AUC}(B)$. Se calculan intervalos percentiles mediante remuestreo **emparejado y estratificado por clase**: la misma muestra bootstrap de filas se utiliza para ambos modelos, preservando los tamaños originales de las clases.

**Importante:** los intervalos son *condicionales a los modelos previamente seleccionados*. No corrigen el optimismo de selección sobre OOF, la dependencia entre transacciones ni la validación de nivel 2 incompletamente anidada del stacking. No equivalen a una evaluación independiente.

In [ ]:
from pathlib import Path
import io, json, zipfile, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score

ROOT = Path.cwd().resolve()
PROJECT = ROOT if (ROOT / 'ensemble_comparison.ipynb').exists() else ROOT / 'final_project'
if not (PROJECT / 'ensemble_comparison.ipynb').exists():
    raise FileNotFoundError('Inicia Jupyter desde el directorio final_project/ o la raíz del repositorio.')
HERE = PROJECT / 'experiments' / 'experiment010'
sys.path.insert(0, str(HERE))
from bootstrap_core import paired_stratified_bootstrap
OUT = PROJECT / 'outputs' / 'bootstrap_comparison'
OUT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='deep')
N_BOOTSTRAP = 600
SEED = 20261123
print('Proyecto:', PROJECT)


## 1. Recuperar predicciones guardadas y etiquetas

Se busca primero la carpeta `outputs/ensemble_extension/` del proyecto; también se admite el ZIP privado de resultados. Para el modelo ajustado por búsqueda se busca el archivo original `all_candidate_predictions.npz`. Nunca utilizamos las etiquetas del conjunto `test.csv`.

In [ ]:
def find_file(name):
    candidates = [PROJECT/'outputs'/name, PROJECT.parent/'outputs'/name,
                  PROJECT/name, PROJECT.parent/name]
    return next((p for p in candidates if p.exists()), None)

prediction_file=find_file('ensemble_extension/five_way_oof_and_test_predictions.npz')
if prediction_file is not None:
    arrays=np.load(prediction_file,allow_pickle=False)
else:
    zip_candidate=find_file('ensemble_extension.zip')
    if zip_candidate is None:
        raise FileNotFoundError('Falta outputs/ensemble_extension/five_way_oof_and_test_predictions.npz o ensemble_extension.zip')
    with zipfile.ZipFile(zip_candidate) as z:
        arrays=np.load(io.BytesIO(z.read('ensemble_extension/five_way_oof_and_test_predictions.npz')))

labels=None
for train_zip in [PROJECT/'playground-series-s6e11.zip',PROJECT.parent/'playground-series-s6e11.zip',
                  PROJECT/'train.csv',PROJECT.parent/'train.csv']:
    if train_zip.exists():
        if train_zip.suffix=='.zip':
            with zipfile.ZipFile(train_zip) as z:
                with z.open('train.csv') as f: labels=pd.read_csv(f,usecols=['is_fraud'])['is_fraud'].astype('uint8').to_numpy()
        else:
            labels=pd.read_csv(train_zip,usecols=['is_fraud'])['is_fraud'].astype('uint8').to_numpy()
        break
if labels is None: raise FileNotFoundError('Falta el ZIP original playground-series-s6e11.zip (o train.csv).')

names = {
'best_single': 'XGBoost individual (prof. 5, tasa 0.05)',
'ensemble1_blend': 'Promedio de probabilidades',
'ensemble2_logit_offset': 'Logit corregido con XGBoost',
'ensemble3_explicit_residual': 'XGBoost sobre residuos de logit',
'ensemble4_rf_stacking': 'Stacking con bosque aleatorio',
}
pred={names[k]:arrays[f'{k}__oof'] for k in names}
original=find_file('all_candidate_predictions.npz')
if original is not None:
    with np.load(original,allow_pickle=False) as cached:
        if 'xgb_nested_grid__oof' in cached.files:
            pred['XGBoost ajustado con GridSearchCV']=np.array(cached['xgb_nested_grid__oof'],copy=True)
        else: print('El archivo original existe, pero no contiene xgb_nested_grid__oof.')
else:
    print('No se encontraron las predicciones del XGBoost de GridSearchCV; se compararán los cinco métodos recuperados.')
assert all(len(v)==len(labels) and np.isfinite(v).all() for v in pred.values())
assert set(np.unique(labels))=={0,1}
print(f'{len(labels):,} observaciones; {labels.sum():,} fraudes; {len(pred)} modelos')


## 2. Verificar los valores originales

La referencia principal es **XGBoost individual (profundidad 5, tasa de aprendizaje 0.05)**, porque fue el mejor de los modelos individuales en la comparación agregada. No debe confundirse con el candidato **XGBoost ajustado mediante `GridSearchCV`**, que se añadirá únicamente si sus predicciones están disponibles.

In [ ]:
reference='XGBoost individual (prof. 5, tasa 0.05)'
observed=pd.DataFrame([{'modelo':k,'ROC AUC':roc_auc_score(labels,v),
                        'Average precision':average_precision_score(labels,v),
                        'Delta AUC':roc_auc_score(labels,v)-roc_auc_score(labels,pred[reference])}
                       for k,v in pred.items()]).sort_values('ROC AUC',ascending=False)
display(observed.style.format({'ROC AUC':'{:.6f}','Average precision':'{:.6f}','Delta AUC':'{:+.6f}'}))


## 3. Bootstrap pareado y estratificado

En cada réplica se extraen con reemplazo $n_1$ fraudes y $n_0$ operaciones legítimas. Se asigna a cada fila un peso igual al número de veces que fue seleccionada. Se calcula el ROC AUC ponderado exactamente, teniendo en cuenta los empates en las probabilidades:

$$
\widehat{\mathrm{AUC}}_b^{(m)}=\frac{\sum_{i:y_i=1}\sum_{j:y_j=0}w_{bi}w_{bj}\bigl[\mathbf{1}(s_i^{(m)}>s_j^{(m)})+\tfrac12\mathbf{1}(s_i^{(m)}=s_j^{(m)})\bigr]}{\left(\sum_{i:y_i=1}w_{bi}\right)\left(\sum_{j:y_j=0}w_{bj}\right)}.
$$

La diferencia en la réplica $b$ es $\Delta_b=\widehat{\mathrm{AUC}}_b^{(m)}-\widehat{\mathrm{AUC}}_b^{(\mathrm{ref})}$. Reutilizar los **mismos pesos** es lo que hace emparejada la comparación; usar bootstrap separado para cada modelo inflaría innecesariamente la incertidumbre.

In [ ]:
summary, draws, differences = paired_stratified_bootstrap(
    labels, pred, reference=reference, n_replicates=N_BOOTSTRAP, seed=SEED,
    progress=lambda i,n: print(f'Remuestreo {i}/{n}')
)
summary.to_csv(OUT/'bootstrap_auc_summary.csv',index=False)
draws.to_csv(OUT/'bootstrap_auc_draws.csv',index=False)
differences.to_csv(OUT/'bootstrap_auc_differences.csv',index=False)
display(summary[['name','roc_auc','delta_vs_reference','delta_ci_low','delta_ci_high','bootstrap_frac_delta_positive']].style.format({
    'roc_auc':'{:.6f}', 'delta_vs_reference':'{:+.6f}', 'delta_ci_low':'{:+.6f}',
    'delta_ci_high':'{:+.6f}', 'bootstrap_frac_delta_positive':'{:.3f}'}))


## 4. Intervalos y diferencias entre modelos

Las barras representan intervalos percentiles del 95% para diferencias en AUC respecto al modelo individual. Una barra que atraviesa cero indica que la muestra no permite afirmar una diferencia distinta de cero **bajo los supuestos del remuestreo**.

In [ ]:
plot=summary.loc[summary['name']!=reference].sort_values('delta_vs_reference').copy()
fig, ax=plt.subplots(figsize=(11,5.3))
colors=sns.color_palette('deep',len(plot))
for i,(_,row) in enumerate(plot.iterrows()):
    x=row['delta_vs_reference']
    ax.errorbar(x,i,xerr=[[max(0,x-row['delta_ci_low'])],[max(0,row['delta_ci_high']-x)]],
                fmt='o',color=colors[i],elinewidth=2.1,capsize=4,markersize=7)
ax.axvline(0,c='#4C72B0',ls='--',lw=1.5,label='Sin diferencia')
ax.set_yticks(range(len(plot)),plot['name'])
ax.set_xlabel('Diferencia de ROC AUC frente a XGBoost individual')
ax.set_title(f'Bootstrap emparejado y estratificado (B={N_BOOTSTRAP:,})')
ax.legend(loc='lower right')
fig.tight_layout();fig.savefig(OUT/'bootstrap_paired_auc.png',dpi=170,bbox_inches='tight');plt.show()


## 5. Decisión estadística y limitaciones

- La selección inicial corresponde a la mayor ROC AUC OOF. El bootstrap cuantifica la **variación por remuestreo de transacciones** para los modelos ya entrenados y elegidos.
- Como los modelos se eligieron con las mismas etiquetas, un intervalo que excluyera cero tampoco garantizaría superioridad fuera de muestra; existe optimismo por selección y múltiples comparaciones.
- Las transacciones podrían no ser independientes por cliente, tarjeta o tiempo. En tal caso sería preferible un bootstrap por grupos o bloques temporales, para lo cual se requieren identificadores adecuados.
- Para comparar contra el XGBoost elegido por `GridSearchCV`, copia `all_candidate_predictions.npz` a `outputs/`; el notebook añadirá automáticamente `xgb_nested_grid__oof` y recalculará todo.
- Los resultados se guardan en `outputs/bootstrap_comparison/` para documentarlos y actualizar las presentaciones sin repetir los modelos.

In [ ]:
metadata={'seed':SEED,'n_bootstrap':N_BOOTSTRAP,'reference':reference,
          'models':list(pred),'n_rows':int(len(labels)),'n_positive':int(labels.sum()),
          'method':'stratified paired nonparametric percentile bootstrap (exact tied weighted AUC)',
          'limitations':['OOF selection optimism','non-fully-nested stacking','potential transaction dependence','exploratory CI']}
(OUT/'bootstrap_manifest.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf8')
print('Archivos guardados en:',OUT)
